# NB09 - Data Lineage & Influence Functions
## Background
Which training examples are responsible for a model's prediction on a specific test sample? Influence functions (Koh & Liang, 2017) answer this by approximating the effect of upweighting each training example on the test loss. Intuitively, removing a harmful training example would change the model's prediction on a test example — influence functions estimate this without actually retraining.

Data lineage tracks where training data comes from, how it was processed, and which model behaviors can be traced to which sources. This is critical for compliance (GDPR right-to-erasure requires knowing which training data affects which model outputs), debugging (tracing hallucinations back to data sources), and safety (identifying which data sources introduce harmful behaviors).

## What You'll Learn
- Influence functions: the I_up,loss formula and first-order approximation
- Identifying most helpful and most harmful training examples
- Data provenance tracking: annotating data with source metadata
- Leave-one-out retraining approximation vs influence estimate

## Why This Matters
LLM training increasingly requires data attribution for copyright and privacy compliance. TracIn (Pruthi et al., 2020) and related methods are used at Google and Anthropic for understanding model behavior through the lens of training data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict
from dataclasses import dataclass, field

# ── Data provenance tracking ───────────────────────────────────────────────
@dataclass
class DataPoint:
    idx: int
    source: str
    collection_date: str
    quality_score: float
    features: np.ndarray
    label: int

@dataclass
class DataLineage:
    source_counts: Dict[str, int] = field(default_factory=dict)
    quality_by_source: Dict[str, List[float]] = field(default_factory=dict)

    def add(self, dp: DataPoint):
        self.source_counts[dp.source] = self.source_counts.get(dp.source, 0) + 1
        if dp.source not in self.quality_by_source:
            self.quality_by_source[dp.source] = []
        self.quality_by_source[dp.source].append(dp.quality_score)

    def report(self):
        print('=== Data Lineage Report ===')
        total = sum(self.source_counts.values())
        print(f'Total samples: {total}')
        for src, count in self.source_counts.items():
            q_scores = self.quality_by_source[src]
            print(f'  {src:20s}: {count:5d} samples, '
                  f'avg quality={np.mean(q_scores):.2f}')

# ── Generate dataset with provenance ──────────────────────────────────────
np.random.seed(42)
n_samples, n_classes, n_features = 500, 4, 6

sources = [
    ('web_scraped',    0.65, 0.2),  # (source, quality_mean, quality_std)
    ('expert_labeled', 0.95, 0.05),
    ('crowdsourced',   0.75, 0.15),
    ('synthetic',      0.80, 0.10),
]

class_means = np.random.randn(n_classes, n_features) * 2
data_points = []
X_all, y_all = [], []

for i in range(n_samples):
    true_class = i % n_classes
    src_name, q_mean, q_std = sources[i % len(sources)]
    quality = float(np.clip(np.random.normal(q_mean, q_std), 0, 1))
    # Low quality -> more noise
    noise_level = (1 - quality) * 2
    feats = class_means[true_class] + np.random.randn(n_features) * (1 + noise_level)
    # Noisy labels for low quality
    label = true_class if np.random.rand() < quality else np.random.randint(n_classes)
    dp = DataPoint(i, src_name, '2024-01', quality, feats, label)
    data_points.append(dp)
    X_all.append(feats); y_all.append(label)

X_all, y_all = np.vstack(X_all), np.array(y_all)

lineage = DataLineage()
for dp in data_points:
    lineage.add(dp)
lineage.report()

In [ ]:
# ── Influence functions (first-order approximation) ────────────────────────
class LinearModel:
    """Simple softmax classifier for influence function demo."""
    def __init__(self, n_features, n_classes):
        np.random.seed(42)
        self.W = np.zeros((n_features, n_classes))
        self.b = np.zeros(n_classes)

    def softmax(self, X):
        z = X @ self.W + self.b
        z -= z.max(axis=1, keepdims=True)
        e = np.exp(z); return e / e.sum(axis=1, keepdims=True)

    def fit(self, X, y, n_epochs=300, lr=0.05, wd=0.01):
        for _ in range(n_epochs):
            probs = self.softmax(X)
            n = len(y)
            dz = probs.copy(); dz[np.arange(n), y] -= 1; dz /= n
            self.W -= lr * (X.T @ dz + wd * self.W)
            self.b -= lr * dz.sum(axis=0)

    def grad_loss(self, X, y):
        """Gradient of cross-entropy loss w.r.t. model parameters."""
        probs = self.softmax(X)
        n = len(y)
        dz = probs.copy(); dz[np.arange(n), y] -= 1; dz /= n
        dW = X.T @ dz
        db = dz.sum(axis=0)
        return np.concatenate([dW.flatten(), db])

# Train model
n_train = int(0.8 * n_samples)
X_tr, y_tr = X_all[:n_train], y_all[:n_train]
X_te, y_te = X_all[n_train:], y_all[n_train:]

model = LinearModel(n_features, n_classes)
model.fit(X_tr, y_tr)

# ── Simplified influence: gradient similarity ─────────────────────────────
# TracIn approximation: influence(z_train, z_test) ~ sum_t grad_train . grad_test
def tracin_influence(model, X_tr, y_tr, X_test_single, y_test_single):
    """Approximate influence of each training example on a single test example."""
    test_grad = model.grad_loss(X_test_single.reshape(1,-1),
                                np.array([y_test_single]))
    influences = []
    for i in range(len(X_tr)):
        train_grad = model.grad_loss(X_tr[i:i+1], y_tr[i:i+1])
        # Influence = dot product of gradients (positive = helpful, negative = harmful)
        influence = float(np.dot(train_grad, test_grad))
        influences.append(influence)
    return np.array(influences)

# Pick a test example to analyze
test_ex_idx = 0
influences = tracin_influence(model, X_tr, y_tr, X_te[test_ex_idx], y_te[test_ex_idx])

top_helpful = np.argsort(influences)[-5:][::-1]
top_harmful = np.argsort(influences)[:5]

print(f'Influence analysis for test sample {test_ex_idx} (true class={y_te[test_ex_idx]})')
print('Most helpful training examples:')
for idx in top_helpful:
    src = data_points[idx].source
    print(f'  train[{idx:3d}] influence={influences[idx]:+.4f}, '
          f'label={y_tr[idx]}, source={src}')
print('Most harmful training examples:')
for idx in top_harmful:
    src = data_points[idx].source
    print(f'  train[{idx:3d}] influence={influences[idx]:+.4f}, '
          f'label={y_tr[idx]}, source={src}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(influences, bins=50, color='steelblue', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', label='Zero influence')
axes[0].set_title('Training Influence Distribution')
axes[0].set_xlabel('TracIn influence score'); axes[0].set_ylabel('Count')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Influence by source
source_names = [dp.source for dp in data_points[:n_train]]
unique_sources = list(set(source_names))
source_influences = {
    s: [influences[i] for i in range(n_train) if source_names[i] == s]
    for s in unique_sources
}

axes[1].boxplot([source_influences[s] for s in unique_sources],
                labels=unique_sources, patch_artist=True)
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Influence Distribution by Data Source')
axes[1].set_xlabel('Data source'); axes[1].set_ylabel('Influence score')
axes[1].tick_params(axis='x', rotation=30)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'{BASE}/s29_09_influence.png', dpi=100, bbox_inches='tight')
plt.show()
print('Expert-labeled data should have highest positive influence; web-scraped lowest.')